In [1]:
import pandas as pd

df = pd.read_csv("customer_churn.csv")
print(df.shape)
print(df.head())
print(df['churn'].value_counts(normalize=True))

(2000, 13)
  customer_id  tenure_months  monthly_charges  total_charges   contract_type  \
0  CUST-00001             11            43.05         535.08        One year   
1  CUST-00002             72            44.33        3124.70  Month-to-month   
2  CUST-00003             31            59.34        1916.12  Month-to-month   
3  CUST-00004             21            74.18        1576.10  Month-to-month   
4  CUST-00005              4            87.84         365.52  Month-to-month   

  internet_service tech_support    payment_method  num_support_calls  \
0              DSL          NaN  Electronic check                  2   
1              DSL          Yes       Credit card                  1   
2      Fiber optic          Yes     Bank transfer                  2   
3              DSL           No      Mailed check                  1   
4              DSL          Yes  Electronic check                  0   

   senior_citizen partner paperless_billing  churn  
0               0     

In [2]:
print(df.isnull().sum())


customer_id           0
tenure_months         0
monthly_charges       0
total_charges        60
contract_type         0
internet_service      0
tech_support         40
payment_method        0
num_support_calls     0
senior_citizen        0
partner               0
paperless_billing     0
churn                 0
dtype: int64


In [3]:
print(df.dtypes)

customer_id              str
tenure_months          int64
monthly_charges      float64
total_charges        float64
contract_type            str
internet_service         str
tech_support             str
payment_method           str
num_support_calls      int64
senior_citizen         int64
partner                  str
paperless_billing        str
churn                  int64
dtype: object


In [4]:
print(df.groupby('contract_type')['churn'].mean())

contract_type
Month-to-month    0.723708
One year          0.404661
Two year          0.399015
Name: churn, dtype: float64


In [5]:
print(df.groupby('tech_support')['churn'].mean())

tech_support
No     0.627065
Yes    0.496372
Name: churn, dtype: float64


In [6]:
print(df.groupby('num_support_calls')['churn'].mean())

num_support_calls
0    0.502174
1    0.536082
2    0.614907
3    0.695279
4    0.743119
5    0.827586
6    0.833333
7    1.000000
Name: churn, dtype: float64


In [7]:
df = df.drop(columns=['customer_id'])

In [8]:
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())
df['tech_support'] = df['tech_support'].fillna('Unknown')

In [9]:
print(df.isnull().sum())

tenure_months        0
monthly_charges      0
total_charges        0
contract_type        0
internet_service     0
tech_support         0
payment_method       0
num_support_calls    0
senior_citizen       0
partner              0
paperless_billing    0
churn                0
dtype: int64


In [10]:
df_encoded = pd.get_dummies(df, columns=['contract_type', 'internet_service', 'tech_support', 'payment_method', 'partner', 'paperless_billing'], drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

(2000, 17)
['tenure_months', 'monthly_charges', 'total_charges', 'num_support_calls', 'senior_citizen', 'churn', 'contract_type_One year', 'contract_type_Two year', 'internet_service_Fiber optic', 'internet_service_No', 'tech_support_Unknown', 'tech_support_Yes', 'payment_method_Credit card', 'payment_method_Electronic check', 'payment_method_Mailed check', 'partner_Yes', 'paperless_billing_Yes']


In [11]:
from sklearn.model_selection import train_test_split

X =df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:

print(X_train.shape,X_test.shape)

(1600, 16) (400, 16)


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

train_accuracy = model.score(X_train,y_train)
test_accuracy = model.score(X_test,y_test)

print(f"Train accuracy :{train_accuracy:.4f}")
print(f"Test accuracy:{test_accuracy:.4f}")

Train accuracy :0.7131
Test accuracy:0.7175


/Users/swapnilshah/Desktop/ai-engineer-roadmap/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

train_accuracy = model.score(X_train_scaled, y_train)
test_accuracy = model.score(X_test_scaled, y_test)

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Train accuracy: 0.7125
Test accuracy: 0.7225


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred)
print(cm)

print(classification_report(y_test, y_pred))

[[100  59]
 [ 52 189]]
              precision    recall  f1-score   support

           0       0.66      0.63      0.64       159
           1       0.76      0.78      0.77       241

    accuracy                           0.72       400
   macro avg       0.71      0.71      0.71       400
weighted avg       0.72      0.72      0.72       400

